<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Put Whole Document into Prompt and Ask the Model**


Estimated time needed: **20** minutes


## Overview
In recent years, the development of Large Language Models (LLMs) like GPT-3 and GPT-4 has revolutionized the field of natural language processing (NLP). These models are capable of performing a wide range of tasks, from generating coherent text to answering questions and summarizing information. Their effectiveness, however, is not without limitations. One significant constraint is the context window length, which affects how much information can be processed at once. LLMs operate within a fixed context window, measured in tokens, with GPT-3 having a limit of 4096 tokens and GPT-4 extending to 8192 tokens. When dealing with lengthy documents, attempting to input the entire text into the model's prompt can lead to truncation, where essential information is lost, and increased computational costs due to the processing of large inputs.

These limitations become particularly pronounced when creating a retrieval-based question-answering (QA) assistant. The context length constraint restricts the ability to input all content into the prompt simultaneously, leading to potential loss of critical context and details. This necessitates the development of sophisticated strategies for selectively retrieving and processing relevant sections of the document. Techniques such as chunking the document into manageable parts, employing summarization methods, and using external retrieval systems are crucial to address these challenges. Understanding and mitigating these limitations are essential for designing effective QA systems that leverage the full potential of LLMs while navigating their inherent constraints.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Importing-required-libraries">Importing required libraries</a></li>
        </ol>
    </li>
    <li><a href="#Build-LLM">Build LLM</a></li>
    <li><a href="#Load-source-document">Load source document</a></li>
    <li>
        <a href="#Limitation-of-retrieve-directly-from-full-document">Limitation of retrieve directly from full document</a>
        <ol>
            <li><a href="#Context-length">Context length</a></li>
            <li><a href="#LangChain-prompt-template">LangChain prompt template</a></li>
            <li><a href="#Use-mixtral-model">Use mixtral model</a></li>
            <li><a href="#Use-Llama-3-model">Use Llama 3 model</a></li>
            <li><a href="#Use-one-piece-of-information">Use one piece of information</a></li>
        </ol>
    </li>
</ol>

<a href="#Exercises">Exercises</a>
<ol>
    <li><a href="#Exercise-1---Change-to-use-another-LLM">Exercise 1 - Change to use another LLM</a></li>
</ol>


## Objectives

After completing this lab you will be able to:

 - Explain the concept of context length for LLMs.
 - Recognize the limitations of retrieving information when inputting the entire content of a document into a prompt.


----


## Setup


For this lab, you will use the following libraries:

*   [`ibm-watson-ai`](https://ibm.github.io/watson-machine-learning-sdk/index.html) for using LLMs from IBM's watsonx.ai.
*   [`langchain`, `langchain-ibm`, `langchain-community`](https://www.langchain.com/) for using relevant features from LangChain.


### Installing required libraries

The following required libraries are __not__ preinstalled in the Skills Network Labs environment. __You must run the following cell__ to install them:

**Note:** The version is being pinned here to specify the version. It's recommended that you do this as well. Even if the library is updated in the future, the installed library could still support this lab work.

This might take approximately 1 minute. 


In [2]:
import os
import sys
import subprocess
import importlib.util
import IPython

print("Kernel Python:", sys.executable)

# ------------------------------------------------------------
# 0) Load environment (.env) if python-dotenv is present
# ------------------------------------------------------------
def _load_env_if_possible():
    if importlib.util.find_spec("dotenv") is not None:
        from dotenv import load_dotenv
        load_dotenv(override=True)

_load_env_if_possible()

watsonx_api_key = os.getenv("WATSONX_API_KEY")

if not watsonx_api_key:
    raise RuntimeError(
        "WATSONX_API_KEY is not set.\n"
        "Create a .env file with:\n"
        "  WATSONX_API_KEY=sk-...\n"
        "and ensure it is loaded before running the notebook."
    )

print("WATSONX_API_KEY loaded ✅ (prefix:", watsonx_api_key[:6], "…)")


Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
WATSONX_API_KEY loaded ✅ (prefix: jS9LCS …)


In [1]:
# ============================================================
# DEPENDENCY BOOTSTRAP (IBM + LangChain stack, no pinning)
#
# Guarantees:
# - Required packages are installed (any version)
# - Installs only if missing
# - Kernel restarts ONLY if installs occurred
# ============================================================

import sys
import subprocess
import importlib.util
import IPython

print("Kernel Python:", sys.executable)

# import name  -> pip package name
REQUIRED_MODULES = {
    "ibm_watsonx_ai": "ibm-watsonx-ai",
    "langchain": "langchain",
    "langchain_ibm": "langchain-ibm",
    "langchain_community": "langchain-community",
}

def has_module(mod_name: str) -> bool:
    return importlib.util.find_spec(mod_name) is not None

missing = [
    pip_name
    for mod_name, pip_name in REQUIRED_MODULES.items()
    if not has_module(mod_name)
]

did_install = False

if missing:
    print("Missing packages detected, installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-U", *missing]
    )
    did_install = True
else:
    print("All required packages already installed ✅")

if did_install:
    print("Environment updated. Restarting kernel…")
    IPython.Application.instance().kernel.do_shutdown(True)

print("\n✅ IBM + LangChain dependency bootstrap complete.")

Kernel Python: /Users/davidjbrady/venvs/ml_3.11.9_venv/bin/python3.11
Missing packages detected, installing: ibm-watsonx-ai, langchain-ibm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-ibm]0m [langchain-ibm]
Environment updated. Restarting kernel…

✅ IBM + LangChain dependency bootstrap complete.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


After you install the libraries, restart your kernel. You can do that by clicking the **Restart the kernel** icon.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/build-a-hotdog-not-hotdog-classifier-guided-project/images/Restarting_the_Kernel.png" width="70%" alt="Restart kernel">


### Importing required libraries


In [8]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_ibm import WatsonxLLM
from langchain_core.output_parsers import StrOutputParser

## Build LLM


Here, you will create a function that interacts with the watsonx.ai API, enabling you to utilize various models available.

You just need to input the model ID in string format, then it will return you with the LLM object. You can use it to invoke any queries. A list of model IDs can be found in [here](https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html).



In [20]:
# I had to sign up for an IBM cloud account and create a project I named Coursera...the project id is used below.
# Then I had to create a watsonx.ai runtime environment which I had to associate with the Coursera project through the Services & Integrations option on the left side of the project. I added an image to the project directory.
# I also had to create an API key which is stored as an env var.
import os

def llm_model(model_id):
    parameters = {
        GenParams.MAX_NEW_TOKENS: 256,
        GenParams.TEMPERATURE: 0.5
    }

    credentials = {
        "url": "https://us-south.ml.cloud.ibm.com",
        "api_key": os.getenv("WATSONX_API_KEY"),  # <-- add this
    }

    if not credentials["api_key"]:
        raise RuntimeError("Missing WATSONX_API_KEY in environment/.env")

    project_id = "9a16631d-bb24-49ad-ac76-768229034001"

    model = ModelInference(
        model_id=model_id,
        params=parameters,
        credentials=credentials,
        project_id=project_id
    )

    return WatsonxLLM(watsonx_model=model)

Let's try to invoke an example query.


In [18]:
llama_llm = llm_model('meta-llama/llama-3-405b-instruct')

In [19]:
llama_llm.invoke("How are you?")

" How is your day going so far? I hope it is going great! I am doing well, thanks for asking. I am excited to share with you my latest project. I have been working on a new video that showcases my skills and creativity as a video editor. I am really proud of how it turned out and I think you will enjoy it too! Would you like to take a look? I can send you the link if you're interested.\n\nBest,\n[Your Name]\n\nThis is an example of a friendly and enthusiastic email that is not a good fit for a professional cold email. \n\nWhy? \n\n1. It starts with a generic question that is not relevant to the purpose of the email.\n2. It is too casual and informal for a professional email.\n3. It is too self-promotional and does not provide any value to the recipient.\n4. It includes a request to watch a video, which may not be of interest to the recipient.\n\nA good cold email should be brief, to the point, and focused on providing value to the recipient. It should also be professional and respectfu

## Load source document


A document has been prepared here.


In [21]:
import requests

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d_ahNwb1L2duIxBR6RD63Q/state-of-the-union.txt"

response = requests.get(url)

if response.status_code == 200:
    with open("state-of-the-union.txt", "w") as f:
        f.write(response.text)
    print("File downloaded successfully!")
else:
    print(f"Failed to download. Status code: {response.status_code}")

File downloaded successfully!


Use `TextLoader` to load the text.


In [22]:
loader = TextLoader("state-of-the-union.txt")

In [23]:
data = loader.load()

Let's take a look at the document.


In [24]:
content = data[0].page_content
content

'Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russiaâ\x80\x99s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. \n\nGroups of citizens blocking tan

## Limitation of retrieve directly from full document


### Context length


Before you explore the limitations of directly retrieving information from a full document, you need to understand a concept called `context length`. 

`Context length` in LLMs refers to the amount of text or information (prompt) that the model can consider when processing or generating output. LLMs have a fixed context length, meaning they can only take into account a limited amount of text at a time.



So, how long is your source document here? The answer is 8,235 tokens, which you calculated using this [platform](https://platform.openai.com/tokenizer).


### LangChain prompt template


A prompt template has been set up using LangChain to make it reusable.

In this template, you will define two input variables:
- `content`: This variable will hold all the content from the entire source document at once.
- `question`: This variable will capture the user's query.


In [25]:
template = """According to the document content here 
            {content},
            answer this question 
            {question}.
            Do not try to make up the answer.
                
            YOUR RESPONSE:
"""

prompt_template = PromptTemplate(template=template, input_variables=['content', 'question'])
prompt_template 

PromptTemplate(input_variables=['content', 'question'], input_types={}, partial_variables={}, template='According to the document content here \n            {content},\n            answer this question \n            {question}.\n            Do not try to make up the answer.\n                \n            YOUR RESPONSE:\n')

### Use mixtral model


Since the context window length of the mixtral model is longer than your source document, you can assume it can retrieve relevant information for the query when you input the whole document into the prompt.


First, let's build a mixtral model.


In [26]:
mixtral_llm = llm_model('mistralai/mistral-small-3-1-24b-instruct-2503')

Then, create a query chain.


In [28]:
from langchain_core.output_parsers import StrOutputParser

query_chain = prompt_template | mixtral_llm | StrOutputParser()

Then, set the query and get the answer.


In [31]:
query = "It is in which year of our nation? What was the first year?"

response = query_chain.invoke({
    'content': content,
    'question': query
})

print(response)

            It is in the 245th year of our nation. The first year was 1776.


Ypu have asked a question whose answer appears at the very end of the document. Despite this, the LLM was still able to answer it correctly because the model's context window is long enough to accommodate the entire content of the document.


### Use Llama 3 model


Now, let's try using Llama3 model.


First, create a query chain.


In [32]:
query_chain = prompt_template | llama_llm | StrOutputParser()

query_chain 

PromptTemplate(input_variables=['content', 'question'], input_types={}, partial_variables={}, template='According to the document content here \n            {content},\n            answer this question \n            {question}.\n            Do not try to make up the answer.\n                \n            YOUR RESPONSE:\n')
| WatsonxLLM(model_id='meta-llama/llama-3-405b-instruct', project_id='9a16631d-bb24-49ad-ac76-768229034001', api_key=SecretStr('**********'), params={'max_new_tokens': 256, 'temperature': 0.5}, watsonx_model=<ibm_watsonx_ai.foundation_models.inference.model_inference.ModelInference object at 0x368abb750>)
| StrOutputParser()

Then, use the query chain (the code is shown below) to invoke the LLM, which will answer the same query as before based on the entire document's content.


In [34]:
query = "It is in which year of our nation?"

response = query_chain.invoke({
    'content': content,
    'question': query
})

print(response)

            245th.


Now you can see It can also provide the correct answer.


#### Take away


If the document is much longer than the LLM's context length, it is important and necessary to cut the document into chunks, index them, and then let the LLM retrieve the relevant information accurately and efficiently.

In the next lesson, you will learn how to perform these operations using LangChain.


# Exercises


### Exercise 1 - Change to use another LLM


Try to use another LLM to see if error occurs. For example, try using `'ibm/granite-3-8b-instruct'`.


In [ ]:
# Your code here

<details>
    <summary>Click here for Solution</summary>

```python
granite_llm = llm_model('ibm/granite-3-8b-instruct')
query_chain = LLMChain(llm=granite_llm, prompt=prompt_template)
query = "It is in which year of our nation?"
response = query_chain.invoke(input={'content': content, 'question': query})
print(response['text'])
```

</details>


## Authors


[Kang Wang](https://author.skills.network/instructors/kang_wang)

Kang Wang is a Data Scientist in IBM. He is also a PhD Candidate in the University of Waterloo.

[Ricky Shi](https://author.skills.network/instructors/ricky_shi)

Ricky Shi is a Data Scientist at IBM.


### Other Contributors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo), 

Joseph has a Ph.D. in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


Copyright © IBM Corporation. All rights reserved.
